In [ ]:
from datetime import datetime, timedelta
import requests
import time
import pandas as pd
import holidays
from category_encoders import TargetEncoder
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
import xgboost as xgb
import numpy as np
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Functies om pipeline op te stellen

In [ ]:
def fetch_kijkcijfers(start_date, end_date):
    data_list = []
    print(f"Start fetching kijkcijfers van {start_date.date()} tot {end_date.date()}")

    # Loop door elke dag
    current_date = start_date
    while current_date <= end_date:
        datum = f"{current_date.year}-{current_date.month}-{current_date.day}"
        url = f"https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={datum}&reportType=north"
        
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                programma_lijst = data.get('hydra:member', [])
                
                for programma in programma_lijst:
                    try:
                        data_list.append({
                            'dateDiff': programma.get('dateDiff'),
                            'ranking': programma.get('ranking'),
                            'description': programma.get('description'),
                            'channel': programma.get('channel'),
                            'startTime': programma.get('startTime'),
                            'rLength': programma.get('rLength'),
                            'rateInK': programma.get('rateInK'),
                            'live': programma.get('live')
                        })
                        
                    except Exception as e:
                        print(f"Fout bij verwerken programma op {datum}: {e}")
            else:
                print(f"Geen data voor {datum} (HTTP {response.status_code})")
                
        except Exception as e:
            print(f"Fout bij ophalen {datum}: {e}")
        
        current_date += timedelta(days=1)
    
    print(f"Einde fetching kijkcijfers")
    # Maak een dataframe van de data
    df = pd.DataFrame(data_list)
    return df

def fetch_weerdata(start_date, end_date):
    # Locatie voor Vlaanderen (Brussel als centraal punt)
    latitude = 50.8503
    longitude = 4.3517

    # API endpoints voor historische data en forecast
    archive_url = "https://archive-api.open-meteo.com/v1/archive"
    forecast_url = "https://api.open-meteo.com/v1/forecast"

    # Relevante hourly variabelen voor het ML model
    hourly_vars = [
        "temperature_2m",   # Gemiddelde temperatuur per uur
        "weathercode",      # Weertype als code per uur
        "precipitation",    # Totale neerslag per uur
        "rain",             # Regen per uur
        "snowfall",         # Sneeuwval per uur
        "cloudcover",       # Bewolking per uur
        "windspeed_10m"     # Windsnelheid per uur
    ]
    
    # Gemeenschappelijke API parameters
    common_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": hourly_vars,
        "timezone": "Europe/Brussels",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "windspeed_unit": "kmh"
    }
    
    today = datetime.today().date()
    dataframes = []
    
    # Historische data (als start_date < vandaag)
    if start_date.date() < today:
        # Bepaal de einddatum voor historische data (niet later dan gisteren)
        hist_end_date = min(end_date.date(), today - timedelta(days=1))
        params_hist = common_params.copy()
        params_hist.update({
            "start_date": start_date.strftime('%Y-%m-%d'),
            "end_date": hist_end_date.strftime('%Y-%m-%d')
        })
        print(f"Fetching historische weerdata van {params_hist['start_date']} tot {params_hist['end_date']}")
        response = requests.get(archive_url, params=params_hist)
        if response.status_code == 200:
            data = response.json().get("hourly", {})
            df_hist = pd.DataFrame({
                "time": data.get("time", []),
                "temperature": data.get("temperature_2m", []),
                "weather_code": data.get("weathercode", []),
                "precipitation": data.get("precipitation", []),
                "rain": data.get("rain", []),
                "snowfall": data.get("snowfall", []),
                "cloudcover": data.get("cloudcover", []),
                "windspeed": data.get("windspeed_10m", [])
            })
            if not df_hist.empty:
                df_hist['time'] = pd.to_datetime(df_hist['time'])
                df_hist['hour'] = df_hist['time'].dt.hour              # Uur van de dag
                df_hist['day_of_week'] = df_hist['time'].dt.dayofweek      # 0 = maandag, 6 = zondag
                df_hist['month'] = df_hist['time'].dt.month                # Maand (1 t/m 12)
                df_hist['year'] = df_hist['time'].dt.year                  # Jaar
                dataframes.append(df_hist)
        else:
            print(f"Fout bij het ophalen van historische data: {response.status_code}")
            print(response.text)
    
    # Forecast data (als end_date >= vandaag)
    if end_date.date() >= today:
        # Voor forecast gebruiken we data vanaf vandaag (of start_date als deze later is dan vandaag)
        forecast_start = max(start_date, datetime.combine(today, datetime.min.time()))
        params_forecast = common_params.copy()
        params_forecast.update({
            "start_date": forecast_start.strftime('%Y-%m-%d'),
            "end_date": end_date.strftime('%Y-%m-%d')
        })
        print(f"Fetching forecast weerdata van {params_forecast['start_date']} tot {params_forecast['end_date']}")
        response = requests.get(forecast_url, params=params_forecast)
        if response.status_code == 200:
            data = response.json().get("hourly", {})
            df_forecast = pd.DataFrame({
                "time": data.get("time", []),
                "temperature": data.get("temperature_2m", []),
                "weather_code": data.get("weathercode", []),
                "precipitation": data.get("precipitation", []),
                "rain": data.get("rain", []),
                "snowfall": data.get("snowfall", []),
                "cloudcover": data.get("cloudcover", []),
                "windspeed": data.get("windspeed_10m", [])
            })
            if not df_forecast.empty:
                df_forecast['time'] = pd.to_datetime(df_forecast['time'])
                df_forecast['hour'] = df_forecast['time'].dt.hour
                df_forecast['day_of_week'] = df_forecast['time'].dt.dayofweek
                df_forecast['month'] = df_forecast['time'].dt.month
                df_forecast['year'] = df_forecast['time'].dt.year
                dataframes.append(df_forecast)
        else:
            print(f"Fout bij het ophalen van forecast data: {response.status_code}")
            print(response.text)
    
    if dataframes:
        # Combineer en sorteer de dataframes op tijd
        df = pd.concat(dataframes).sort_values("time").reset_index(drop=True)
    else:
        df = pd.DataFrame()
    
    print("Einde fetching weerdata")
    return df

In [ ]:
def fetch_weerdata(start_date, end_date):
    # Locatie voor Vlaanderen (Brussel als centraal punt)
    latitude = 50.8503
    longitude = 4.3517

    # API endpoints voor historische data en forecast
    archive_url = "https://archive-api.open-meteo.com/v1/archive"
    forecast_url = "https://api.open-meteo.com/v1/forecast"

    # Relevante hourly variabelen voor het ML model
    hourly_vars = [
        "temperature_2m",   # Gemiddelde temperatuur per uur
        "weathercode",      # Weertype als code per uur
        "precipitation",    # Totale neerslag per uur
        "rain",             # Regen per uur
        "snowfall",         # Sneeuwval per uur
        "cloudcover",       # Bewolking per uur
        "windspeed_10m"     # Windsnelheid per uur
    ]
    
    # Gemeenschappelijke API parameters
    common_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": hourly_vars,
        "timezone": "Europe/Brussels",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "windspeed_unit": "kmh"
    }
    
    today = datetime.today().date()
    dataframes = []
    
    # Historische data (als start_date < vandaag)
    if start_date.date() < today:
        # Bepaal de einddatum voor historische data (niet later dan gisteren)
        hist_end_date = min(end_date.date(), today - timedelta(days=1))
        params_hist = common_params.copy()
        params_hist.update({
            "start_date": start_date.strftime('%Y-%m-%d'),
            "end_date": hist_end_date.strftime('%Y-%m-%d')
        })
        print(f"Fetching historische weerdata van {params_hist['start_date']} tot {params_hist['end_date']}")
        response = requests.get(archive_url, params=params_hist)
        if response.status_code == 200:
            data = response.json().get("hourly", {})
            df_hist = pd.DataFrame({
                "time": data.get("time", []),
                "temperature": data.get("temperature_2m", []),
                "weather_code": data.get("weathercode", []),
                "precipitation": data.get("precipitation", []),
                "rain": data.get("rain", []),
                "snowfall": data.get("snowfall", []),
                "cloudcover": data.get("cloudcover", []),
                "windspeed": data.get("windspeed_10m", [])
            })
            if not df_hist.empty:
                df_hist['time'] = pd.to_datetime(df_hist['time'])
                df_hist['hour'] = df_hist['time'].dt.hour              # Uur van de dag
                df_hist['day_of_week'] = df_hist['time'].dt.dayofweek      # 0 = maandag, 6 = zondag
                df_hist['month'] = df_hist['time'].dt.month                # Maand (1 t/m 12)
                df_hist['year'] = df_hist['time'].dt.year                  # Jaar
                dataframes.append(df_hist)
        else:
            print(f"Fout bij het ophalen van historische data: {response.status_code}")
            print(response.text)
    
    # Forecast data (als end_date >= vandaag)
    if end_date.date() >= today:
        # Voor forecast gebruiken we data vanaf vandaag (of start_date als deze later is dan vandaag)
        forecast_start = max(start_date, datetime.combine(today, datetime.min.time()))
        params_forecast = common_params.copy()
        params_forecast.update({
            "start_date": forecast_start.strftime('%Y-%m-%d'),
            "end_date": end_date.strftime('%Y-%m-%d')
        })
        print(f"Fetching forecast weerdata van {params_forecast['start_date']} tot {params_forecast['end_date']}")
        response = requests.get(forecast_url, params=params_forecast)
        if response.status_code == 200:
            data = response.json().get("hourly", {})
            df_forecast = pd.DataFrame({
                "time": data.get("time", []),
                "temperature": data.get("temperature_2m", []),
                "weather_code": data.get("weathercode", []),
                "precipitation": data.get("precipitation", []),
                "rain": data.get("rain", []),
                "snowfall": data.get("snowfall", []),
                "cloudcover": data.get("cloudcover", []),
                "windspeed": data.get("windspeed_10m", [])
            })
            if not df_forecast.empty:
                df_forecast['time'] = pd.to_datetime(df_forecast['time'])
                df_forecast['hour'] = df_forecast['time'].dt.hour
                df_forecast['day_of_week'] = df_forecast['time'].dt.dayofweek
                df_forecast['month'] = df_forecast['time'].dt.month
                df_forecast['year'] = df_forecast['time'].dt.year
                dataframes.append(df_forecast)
        else:
            print(f"Fout bij het ophalen van forecast data: {response.status_code}")
            print(response.text)
    
    if dataframes:
        # Combineer en sorteer de dataframes op tijd
        df = pd.concat(dataframes).sort_values("time").reset_index(drop=True)
    else:
        df = pd.DataFrame()
    
    print("Einde fetching weerdata")
    return df